# API-232 Lab (SOLUTIONS): Does Getting the Ball First Win Overtime?
### A randomized experiment run by a coin toss

When an NFL game is tied after four quarters, it goes to overtime. A **coin toss** decides which team chooses first, and the winner almost always chooses to **receive the kickoff** — to get the ball first.

That makes overtime a natural **randomized experiment**:

* **Treatment:** your team gets the ball first in overtime.
* **Control:** the other team gets the ball first.
* **Outcome:** your team wins the game.
* **Assignment mechanism:** a coin flip.

The NFL has changed its overtime rules twice because it worried that the coin toss was deciding too many games:

| Rules | Seasons | How overtime ends |
|---|---|---|
| **Sudden death** | Regular season through 2011; playoffs through the 2009 season | First team to score any points wins. |
| **Modified sudden death** | Regular season 2012–2024; playoffs 2010–2021 | A touchdown on the first possession wins. A first-possession field goal gives the other team one possession. (The regular-season period was cut from 15 to 10 minutes in 2017.) |
| **Both teams get the ball** | Playoffs from 2022; regular season from 2025 | Both teams are guaranteed a possession, whatever happens on the first one. |

In this lab you will use real play-by-play data from every NFL game from 1999 to 2025 to answer:

1. Is getting the ball first an advantage? How big, and how sure are we?
2. Did the 2010/2012 rule change reduce the advantage?
3. Could this data even *detect* the change the NFL was hoping for?

**Concepts covered:** treatment effects, randomization and balance, standard errors, confidence intervals, statistical significance vs. practical importance, power, and non-interference.

**Time:** about 60 minutes. Answer the written questions in the markdown cells marked **✏️ Your answer**.


## 0. Setup (about 3 minutes)

Run the next two cells. The first installs `nflreadpy`, the Python package for the free [nflverse](https://github.com/nflverse) NFL data. The second downloads one season at a time and keeps only overtime plays, so it stays within Colab's memory. **Downloading 27 seasons takes 1–3 minutes.**

In [ ]:
!pip install -q nflreadpy

import math
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import nflreadpy as nfl

pd.set_option("display.precision", 3)
rng = np.random.default_rng(232)  # fixed seed so everyone gets the same simulation results

In [ ]:
COLS = ["game_id", "play_id", "season", "season_type", "qtr", "play_type", "posteam",
        "home_team", "away_team", "home_score", "away_score", "spread_line",
        "fixed_drive", "fixed_drive_result"]

frames = []
for season in range(1999, 2026):
    pbp = nfl.load_pbp(season)
    frames.append(pbp.filter(pl.col("qtr") >= 5).select(COLS).to_pandas())  # qtr 5 = overtime
    print(season, end=" ")

ot_plays = pd.concat(frames, ignore_index=True).sort_values(["game_id", "play_id"])
print(f"\nLoaded {len(ot_plays):,} overtime plays from {ot_plays.game_id.nunique()} games.")

### Building the game-level data (provided)

The cell below turns plays into **one row per overtime game**. The team that **received the opening overtime kickoff** is the team that got the ball first. Read the code and comments, but you don't need to change anything.

Variables you'll use:

* `received_first`: team that got the ball first (**treated** team)
* `first_wins`: 1 if the treated team won the game, 0 if it lost (ties are dropped)
* `era`: which overtime rules were in force
* `recv_home`: 1 if the treated team was the home team
* `recv_spread`: the treated team's expected margin of victory from the pregame betting line (positive = it was favoured)

In [ ]:
# The receiving team on the first overtime kickoff got the ball first.
kick = (ot_plays[ot_plays.play_type == "kickoff"]
        .groupby("game_id", as_index=False).first()[["game_id", "posteam"]]
        .rename(columns={"posteam": "received_first"}))

games = (ot_plays.groupby("game_id", as_index=False)
         .agg(season=("season", "first"), season_type=("season_type", "first"),
              home_team=("home_team", "first"), away_team=("away_team", "first"),
              home_score=("home_score", "max"), away_score=("away_score", "max"),
              spread_line=("spread_line", "first"))   # spread_line > 0 means the home team was favoured
         .merge(kick, on="game_id"))

games["margin"] = games.home_score - games.away_score
games["winner"] = np.select([games.margin > 0, games.margin < 0],
                            [games.home_team, games.away_team], default="TIE")

def rules(row):
    playoffs = row.season_type == "POST"
    if (not playoffs and row.season <= 2011) or (playoffs and row.season <= 2009):
        return "1. Sudden death"
    if (not playoffs and row.season <= 2024) or (playoffs and row.season <= 2021):
        return "2. Modified sudden death"
    return "3. Both teams get the ball"

games["era"] = games.apply(rules, axis=1)
games["recv_home"] = (games.received_first == games.home_team).astype(int)
games["recv_spread"] = np.where(games.recv_home == 1, games.spread_line, -games.spread_line)

print("Ties:", (games.winner == "TIE").sum())
ot = games[games.winner != "TIE"].copy()
ot["first_wins"] = (ot.winner == ot.received_first).astype(int)
print("Overtime games with a winner:", len(ot))
ot.head()

## Part 1. Was the coin toss really random? (about 10 minutes)

Before estimating any effect, we check **balance**: if the toss is random, the treated teams (got the ball first) should look like the control teams on things decided *before* the toss.

We'll check two pre-treatment characteristics:

1. **Home field.** In a random assignment, the home team should get the ball first about 50% of the time.
2. **Team strength.** The pregame point spread tells us which team the betting market expected to win. If the toss is random, the treated team's average expected margin should be about **0**.

**1a.** Compute the share of games in which the home team got the ball first, and the average of `recv_spread`. Report a standard error for each.

In [ ]:
n = len(ot)
home_share = ot.recv_home.mean()
home_se = math.sqrt(home_share * (1 - home_share) / n)

spread = ot.recv_spread.dropna()
spread_mean = spread.mean()
spread_se = spread.std() / math.sqrt(len(spread))

print(f"Home team got the ball first: {home_share:.3f} (SE {home_se:.3f})")
print(f"Receiving team's average expected margin: {spread_mean:+.2f} points (SE {spread_se:.2f})")

**1b.** Are both results consistent with a random toss? Explain using the standard errors.

✅ **Answer:** Yes. The home team got the ball first in about 51% of games, roughly half a standard error (SE ≈ 2.4 points) from 50%. The receiving team's average expected margin is about +0.4 points with an SE of about 0.27, so about 1.6 standard errors from 0. Neither is beyond the ~2 standard errors we'd need to doubt the toss, and +0.4 points is tiny in football terms. This is what randomization predicts: treated and control teams are similar on characteristics fixed before the toss.

**1c.** The data doesn't record *who won the toss*, only *who got the ball first*. When could that difference bias our estimate?

✅ **Answer:** Winning the toss is random; *choosing* to receive is not. Getting the ball first is like *taking up* a program, and winning the toss is like being *offered* it. If toss winners who choose to kick are different (for example, coaches with strong defences, or windy games like the 2002 Lions), then "got the ball first" is no longer purely random. Comparing by it would mix the effect of possession with whatever drove the choice.

The clean comparison is by *toss outcome* (the intention-to-treat effect). Here, kicking after winning the toss is extremely rare, so "got the ball first" and "won the toss" match in nearly every game. The bias is therefore negligible, and the balance checks in 1a support that.

## Part 2. How big is the first-possession advantage? (about 15 minutes)

**2a.** Using all overtime games, compute:

* the share of games won by the team that got the ball first, `p_hat`
* its standard error
* a 95% confidence interval
* the z-score for the null hypothesis that getting the ball first doesn't matter (p = 0.5)
* the two-sided p-value

Use 1.96 for the confidence interval. The two-sided p-value for a z-score is `math.erfc(abs(z) / math.sqrt(2))`.

In [ ]:
p_hat = ot.first_wins.mean()
se = math.sqrt(p_hat * (1 - p_hat) / len(ot))
ci_low, ci_high = p_hat - 1.96 * se, p_hat + 1.96 * se
z = (p_hat - 0.5) / se
p_value = math.erfc(abs(z) / math.sqrt(2))

print(f"First possession wins {p_hat:.3f} of games (n = {len(ot)})")
print(f"SE = {se:.3f}; 95% CI = [{ci_low:.3f}, {ci_high:.3f}]; z = {z:.2f}; p = {p_value:.3f}")

**2b. From a win rate to a treatment effect.**

(i) Write the average treatment effect of getting the ball first on the probability of winning, in terms of p. Compute it.

✅ **Answer:** ATE = P(win | treated) − P(win | control) = p − (1 − p) = **2p − 1**. With p̂ ≈ 0.583, the effect is about **+17 percentage points**: getting the ball first raises a team's chance of winning from about 42% to about 58%. Its standard error is 2 × SE(p̂), about 4.8 points.

(ii) Why is the two-group formula wrong here?

✅ **Answer:** The two groups aren't independent; they are the *same games*. When the treated team wins, the control team loses, so the two outcomes are perfectly negatively correlated. The two-group formula assumes independent samples. This is a failure of **non-interference**: one team's treatment (getting the ball) directly sets the other team's outcome. That's why we analyse **one row per game**, the same reason experiments with spillovers within a household randomize whole households.

**2c.** Which statement correctly interprets your 95% confidence interval?

✅ **Answer: 2.** The true win rate is fixed; the *interval* is what varies from sample to sample. 95% describes how often the procedure captures the truth. (1) treats the true rate as random. (3) mixes up the average with individual games. (4) confuses the interval with the spread of the estimates.

## Part 3. Did the rule change work? (about 15 minutes)

The NFL brought in modified sudden death to reduce the coin toss's influence. Under sudden death, a field goal on the first possession ended the game. Under the new rules it doesn't.

**3a.** Fill in the function below, then use it to build a table by `era` with n, the win rate, SE, 95% CI and z-score. The cell after it plots the results.

In [ ]:
def summarize(x):
    """x is a Series of 0/1 outcomes. Return n, p, SE, 95% CI and the z-score against 0.5."""
    n = len(x)
    p = x.mean()
    se = math.sqrt(p * (1 - p) / n)
    return pd.Series({"n": n, "p": p, "se": se,
                      "ci_low": p - 1.96 * se, "ci_high": p + 1.96 * se,
                      "z": (p - 0.5) / se})

by_era = ot.groupby("era").first_wins.apply(summarize).unstack()
by_era

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
y = np.arange(len(by_era))
ax.errorbar(by_era.p, y, xerr=1.96 * by_era.se, fmt="o", capsize=4)
ax.axvline(0.5, color="grey", ls="--", lw=1)
ax.set_yticks(y, [f"{era}  (n={int(n)})" for era, n in zip(by_era.index, by_era.n)])
ax.invert_yaxis()
ax.set_xlabel("Share of overtime games won by the team that got the ball first")
ax.set_title("First-possession win rate with 95% confidence intervals")
plt.tight_layout(); plt.show()

**3b.** Estimate the change in the first-possession win rate from sudden death to modified sudden death, with a standard error, a 95% CI and a z-score.

*Hint:* the two eras are different games, so here the samples **are** independent: SE(difference) = √(SE₁² + SE₂²).

In [ ]:
sd = by_era.loc["1. Sudden death"]
mod = by_era.loc["2. Modified sudden death"]

diff = mod.p - sd.p
se_diff = math.sqrt(sd.se**2 + mod.se**2)
z_diff = diff / se_diff

print(f"Change: {diff:+.3f} (SE {se_diff:.3f}); 95% CI = [{diff - 1.96*se_diff:+.3f}, {diff + 1.96*se_diff:+.3f}]; z = {z_diff:.2f}")

**3c.** Evaluate the columnist's and the official's claims.

✅ **Answer:** Use the numbers your table printed. In this data the advantage did **not** shrink: the first team won about 58.5% of games under sudden death and about 58.2% under modified sudden death, a change of −0.3 points. Both claims still overreach:

* **Columnist:** whether an era's advantage is significant against 50% says nothing about whether it *changed*. To judge the rule change you have to test the *difference between eras*, which is what 3b does.
* **Official:** a non-significant difference doesn't mean "no change". The confidence interval for the change is wide (about −10 to +9 points). It includes zero, but it also includes a sizeable *drop* or *rise* in the advantage. Absence of evidence isn't evidence of absence; the honest summary is that the data can't tell. Part 4 shows why.

**3d.** What can we conclude from the "Both teams get the ball" era?

✅ **Answer:** Very little yet. There are only 16 decided games, so the standard error is about 12 points and the confidence interval runs from roughly 32% to 81%. It is consistent with no advantage *and* with a large one. These games are also mostly playoff games, which may differ from the regular season (stronger teams, no ties allowed). That's an **external validity** concern if we try to apply the result to the 2025 regular-season rule.

## Part 4. Could we have detected a rule-change effect? (about 12 minutes)

Suppose the rule change *really* cut the first-possession win rate by d percentage points. How likely is a study with our sample sizes to find a statistically significant drop? That probability is the study's **power**.

**4a.** Before simulating anything, use a standard rule of thumb. Significance requires an estimate more than about 2 × SE(difference) from zero. With SE(difference) ≈ your answer from 3b, how big would the true drop need to be for **50% power**? For **80% power** (about 2.8 × SE)?

✏️ **Your answer:**

&nbsp;

**4b.** The simulation below checks your answer. For each possible true drop d, it creates 2,000 fake versions of history. In each one, sudden-death games are won at the observed sudden-death rate, modified games at that rate minus d, using the real number of games in each era. It then counts the share of runs in which a two-sided test at the 5% level detects a significant drop. Fill in the two TODO lines.

In [ ]:
n_sd, n_mod = int(sd.n), int(mod.n)
p_sd = sd.p
drops = np.arange(0, 0.31, 0.02)
power = []

for d in drops:
    p_mod = p_sd - d
    hits = 0
    for _ in range(2000):
        x_sd = rng.binomial(n_sd, p_sd) / n_sd       # simulated sudden-death win rate
        x_mod = rng.binomial(n_mod, p_mod) / n_mod   # simulated modified win rate
        se_sim = math.sqrt(x_sd * (1 - x_sd) / n_sd + x_mod * (1 - x_mod) / n_mod)
        z_sim = (x_mod - x_sd) / se_sim
        hits += z_sim < -1.96   # a significant DROP
    power.append(hits / 2000)

plt.figure(figsize=(6, 3.5))
plt.plot(drops * 100, power, marker="o")
plt.axhline(0.8, color="grey", ls="--", lw=1); plt.axhline(0.5, color="grey", ls=":", lw=1)
plt.xlabel("True drop in first-possession win rate (percentage points)")
plt.ylabel("Power"); plt.title("Power to detect the rule change")
plt.tight_layout(); plt.show()
pd.DataFrame({"true drop (pp)": drops * 100, "power": power}).T.round(2)

**4a.** How big would the true drop need to be for 50% and 80% power?

✅ **Answer:** With SE(difference) ≈ 0.049 (from 3b), **50% power** needs a true drop of about 2 × 0.049 ≈ **10 points**, and **80% power** about 2.8 × 0.049 ≈ **14 points**. The simulation should roughly match this.

**4c.** Power to detect a 5-point drop, and the games needed for 80% power?

✅ **Answer:** Read it from the simulation table: power for a 5-point drop is only about **15–20%** (12% at 4 points, 25% at 6). For 80% power we need the 80%-power drop to fall from about 14 points to 5, so the SE must shrink by a factor of about 14/5 ≈ 2.8. Since SE ∝ 1/√n, that takes about 2.8² ≈ 8 times as many games: **about 1,600–1,700 overtime games per era**, roughly a century of NFL overtime.

**4d.** How does the power analysis change your response to the official?

✅ **Answer:** The comparison was badly underpowered for any realistic effect. A non-significant result was the *likely* outcome even if the rule worked as intended, so it is weak evidence that the rule failed. The right conclusion is "we can't tell", not "it did nothing". Better evidence would come from *mechanisms* with much larger samples, such as how often first drives end in field goals, as in the extension below.

## Part 5. Significance vs. importance (about 5 minutes)

**5.** Respond to the coach and the executive.

✅ **Answer:**
* **Coach:** statistical significance is about how *precise* the estimate is, not whether the effect matters. The insignificant eras have few games, so their confidence intervals are wide. Only the newest era, with 16 games, is insignificant, and pooling all eras gives a clearly significant first-team win rate of about 58% (z ≈ 3.5).
* **Executive:** in a contest meant to be a 50/50 tiebreaker, moving the odds to roughly 58/42 because of a *coin flip* is large. It's comparable to home-field advantage, and in a playoff game it decides a season.

The general lesson: significance doesn't make an effect important, and a lack of significance doesn't make it unimportant. Judge size and precision separately.

## Extension (optional): the mechanism

Under sudden death, a field goal on the first possession won the game. The modified rules took that away. If the rules worked, the **first drive's result** should matter less.

**E1.** Run the cell below. It reports how each era's first overtime drive ended and how often the team that got the ball first won after each result. Under modified sudden death, what happens to teams that kick a field goal on the first drive? Does that support the idea that the rule change shifted *how* teams win rather than *whether* the first team wins?

In [ ]:
first_drive = (ot_plays[ot_plays.fixed_drive.notna()]
               .groupby("game_id", as_index=False).first()[["game_id", "fixed_drive_result"]])
fd = ot.merge(first_drive, on="game_id")
fd["result"] = fd.fixed_drive_result.replace({"Turnover on downs": "Turnover", "Opp touchdown": "Turnover"})
summary = (fd.groupby(["era", "result"]).first_wins.agg(games="size", first_team_win_rate="mean")
           .reset_index())
summary["share_of_era"] = summary.games / summary.groupby("era").games.transform("sum")
summary.sort_values(["era", "games"], ascending=[True, False])

✅ **Answer (discussion):** Under sudden death, a first-drive field goal was an automatic win: 27% of first drives ended that way, and the first team won 100% of them. Under modified sudden death, the first team won only about 70% of games after a first-drive field goal, and field goals fell to about 16% of first drives. First-drive touchdowns, which still win outright, rose from about 7% to about 22% of first drives. Teams adapted by playing for touchdowns, which helps explain why the overall advantage barely moved: the rule changed *how* the first team wins more than *whether* it wins.

These shares are based on **every** overtime game in each era, so they are measured far more precisely than the change in overall win rates. This is a good example of using a mechanism to learn about a policy when the main outcome is underpowered.